# Section title

- `note 1`



In [ ]:
import os

import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import ToTensor

In [ ]:
%load_ext watermark

# Section 1



In [ ]:
DATA_DIR="../local_data/transformer-mentor_data"


100%|██████████| 26.4M/26.4M [00:03<00:00, 7.78MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 189kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.37MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.2MB/s]


# Section 2

In [1]:
%watermark

UsageError: Line magic function `%watermark` not found.


# Stock adjustment

In [ ]:
def rebalance_portfolio(current_amounts, target_percentages):
    """
    Calculates the buy/sell amounts to reach target percentages.
    
    :param current_amounts: dictionary of {fund_name: current_value}
    :param target_percentages: dictionary of {fund_name: target_decimal}
    :return: dictionary of {fund_name: transaction_amount} 
             (Positive = Buy, Negative = Sell)
    """
    total_value = sum(current_amounts.values())
    rebalance_actions = {}

    for fund, current_val in current_amounts.items():
        # Calculate what the fund SHOULD be worth
        target_val = total_value * target_percentages[fund]
        
        # Difference: Target - Current
        # Positive result means we need to add money (Buy)
        # Negative result means we have too much (Sell)
        diff = target_val - current_val
        rebalance_actions[fund] = round(diff, 2)

    return rebalance_actions

# --- Test Case ---
if __name__ == "__main__":
    # Current portfolio state (Total = $10,000)
    current_portfolio = {
        "S&P 500 Index": 6500,
        "International Stock": 2000,
        "Total Bond Market": 1500
    }

    # Target Allocations (Must sum to 1.0)
    targets = {
        "S&P 500 Index": 0.50,     # Target: $5,000
        "International Stock": 0.30, # Target: $3,000
        "Total Bond Market": 0.20    # Target: $2,000
    }

    results = rebalance_portfolio(current_portfolio, targets)

    print(f"{'Fund Name':<20} | {'Action':<10} | {'Amount'}")
    print("-" * 45)
    for fund, amount in results.items():
        action = "BUY" if amount > 0 else "SELL"
        print(f"{fund:<20} | {action:<10} | ${abs(amount):,.2f}")

Fund Name            | Action     | Amount
---------------------------------------------
S&P 500 Index        | SELL       | $1,500.00
International Stock  | BUY        | $1,000.00
Total Bond Market    | BUY        | $500.00


In [ ]:

# --- Actual Case ---
if __name__ == "__main__":
    # Current portfolio state (Total = $10,000)
    current_portfolio = {
        "FBIIX": 9877,
        "FXAIX": 21773,
        "FXNAX": 23000,
        "FZILX": 99000,
        "FZROX": 162904
    }

    # Target Allocations (Must sum to 1.0)
    targets = {
        "FBIIX": 0.50,     # Target: $5,000
        "FXAIX": 0.30, # Target: $3,000
        "FXNAX": 0.20,    # Target: $2,000
        "FZILX": 99000,
        "FZROX": 162904
    }

    results = rebalance_portfolio(current_portfolio, targets)

    print(f"{'Fund Name':<20} | {'Action':<10} | {'Amount'}")
    print("-" * 45)
    for fund, amount in results.items():
        action = "BUY" if amount > 0 else "SELL"
        print(f"{fund:<20} | {action:<10} | ${abs(amount):,.2f}")

Fund Name            | Action     | Amount
---------------------------------------------
FBIIX                | BUY        | $136,900.00
FXAIX                | BUY        | $66,293.20
FXNAX                | BUY        | $47,820,957,912.00
FZILX                | BUY        | $29,061,747,000.00


In [5]:
def rebalance_portfolio(
    current_amounts: dict[str, float], 
    target_percentages: dict[str, float]
) -> dict[str, dict[str, float]]:
    """Calculates rebalancing actions and current portfolio weightings.

    Args:
        current_amounts: A dictionary mapping fund names (str) to their 
            current market value (float).
        target_percentages: A dictionary mapping fund names (str) to their 
            target allocation as a decimal (float, e.g., 0.5 for 50%).

    Returns:
        A dictionary where each key is a fund name and the value is a 
        dictionary containing:
            'current_pct': The current weight of the fund in the portfolio.
            'action_amount': The dollar amount to buy (positive) or 
                sell (negative) to reach the target.
    """
    total_value = sum(current_amounts.values())
    results = {}

    for fund, current_val in current_amounts.items():
        # Calculate current weighting
        current_pct = current_val / total_value if total_value > 0 else 0
        
        # Calculate target dollar value
        target_val = total_value * target_percentages[fund]
        
        # Difference: Target - Current
        diff = target_val - current_val
        
        results[fund] = {
            "current_pct": round(current_pct, 4),
            "action_amount": round(diff, 2)
        }

    return results


In [6]:
# --- Test Case ---
if __name__ == "__main__":
    # Test Data
    portfolio = {
        "US Stock Index": 7200.0,
        "Intl Stock Index": 1800.0,
        "Bond Index": 1000.0
    }
    
    targets = {
        "US Stock Index": 0.60,
        "Intl Stock Index": 0.30,
        "Bond Index": 0.10
    }

    rebalance_data = rebalance_portfolio(portfolio, targets)

    # Displaying the results
    print(f"{'Fund Name':<18} | {'Current %':<10} | {'Target %':<10} | {'Action Amount'}")
    print("-" * 65)
    
    for fund, data in rebalance_data.items():
        curr_pct = data['current_pct'] * 100
        target_pct = targets[fund] * 100
        amount = data['action_amount']
        action = "BUY" if amount > 0 else "SELL"
        
        print(f"{fund:<18} | {curr_pct:>8.1f}% | {target_pct:>8.1f}% | {action} ${abs(amount):,.2f}")

Fund Name          | Current %  | Target %   | Action Amount
-----------------------------------------------------------------
US Stock Index     |     72.0% |     60.0% | SELL $1,200.00
Intl Stock Index   |     18.0% |     30.0% | BUY $1,200.00
Bond Index         |     10.0% |     10.0% | SELL $0.00


In [8]:
# --- Actual Case ---
if __name__ == "__main__":
    current_portfolio = {
        "FBIIX": 9877, #  Fidelity ® International Bond Index Fund
        "FXIAX": 21773, # Fidelity ® 500 Index Fund
        "FXNAX": 23000,   # Fidelity ® U.S. Bond Index Fund
        "FZILX": 99000,  # Fidelity ® ZERO International Index Fund
        "FZROX": 162904 # Fidelity ® ZERO Total Market Index Fund
    }

    # Target Allocations (Must sum to 1.0)
    targets = {
        "FBIIX": 0.04,
        "FXIAX": 0.10,
        "FXNAX": 0.06,
        "FZILX": 0.25,
        "FZROX": 0.55
    }

    rebalance_data = rebalance_portfolio(current_portfolio, targets)
    # Displaying the results
    print(f"{'Fund Name':<18} | {'Current %':<10} | {'Target %':<10} | {'Action Amount'}")
    print("-" * 65)
    
    for fund, data in rebalance_data.items():
        curr_pct = data['current_pct'] * 100
        target_pct = targets[fund] * 100
        amount = data['action_amount']
        action = "BUY" if amount > 0 else "SELL"
        
        print(f"{fund:<18} | {curr_pct:>8.1f}% | {target_pct:>8.1f}% | {action} ${abs(amount):,.2f}")

Fund Name          | Current %  | Target %   | Action Amount
-----------------------------------------------------------------
FBIIX              |      3.1% |      4.0% | BUY $2,785.16
FXIAX              |      6.9% |     10.0% | BUY $9,882.40
FXNAX              |      7.3% |      6.0% | SELL $4,006.76
FZILX              |     31.3% |     25.0% | SELL $19,861.50
FZROX              |     51.5% |     55.0% | BUY $11,200.70


In [5]:
%watermark -iv

torch      : 2.7.1
torchvision: 0.22.1
matplotlib : 3.10.3

